In [1]:
import pandas as pd
import numpy as np
import time
from greedy_auto_time_start_1 import Point
from sklearn.metrics import silhouette_score

### Data preprocessing for experiments

In [2]:
OSM_URL = "https://router.project-osrm.org"
OSM_MODE = "driving"
BLOCK_SIZE = 25
CREW_MEMBERS = 3
MAX_WORK_HOURS = 12

#### Define depot points

In [3]:
depots = pd.read_csv('synthetic_data_kyiv_varash/general.csv')

In [4]:
kyiv = depots[depots['city_name'] == 'Kyiv'].iloc[0]
kyiv_depot = Point("DEPOT", kyiv['depot_lan'], kyiv['depot_lot'], "DEPOT")

In [5]:
varash = depots[depots['city_name'] == 'Varash'].iloc[0]
varash_depot = Point("DEPOT", varash['depot_lan'], varash['depot_lot'], "DEPOT")

In [6]:
depots_simple = pd.read_csv('synthetic_data_ternopil_dubno/general.csv')

In [7]:
ternopil = depots_simple[depots_simple['city_name'] == 'Ternopil'].iloc[0]
ternopil_depot = Point("DEPOT", ternopil['depot_lan'], ternopil['depot_lot'], "DEPOT")

In [8]:
dubno = depots_simple[depots_simple['city_name'] == 'Dubno'].iloc[0]
dubno_depot = Point("DEPOT", dubno['depot_lan'], dubno['depot_lot'], "DEPOT")

#### Define the list of points to be serviced

In [9]:
kyiv_small = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_15.csv')
kyiv_small_1 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_03.csv')
kyiv_meduim_1 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_22.csv')
kyiv_meduim_2 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_11.csv')
kyiv_meduim_3 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_05.csv')
kyiv_meduim_4 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_12.csv')
kyiv_meduim_5 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_25.csv')
kyiv_gross_1 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_02.csv')
kyiv_gross_2 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_23.csv')
kyiv_gross_3 = pd.read_csv('synthetic_data_kyiv_varash/kyiv/kyiv_day_19.csv')

In [10]:
varash_small = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_15.csv')
varash_small_1 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_23.csv')
varash_medium_1 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_10.csv')
varash_medium_2 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_21.csv')
varash_medium_3 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_09.csv')
varash_medium_4 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_24.csv')
varash_medium_5 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_12.csv')
varash_gross_1 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_06.csv')
varash_gross_2 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_02.csv')
varash_gross_3 = pd.read_csv('synthetic_data_kyiv_varash/varash/varash_day_25.csv')

In [11]:
ternopil_small = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_07.csv')
ternopil_small_1 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_25.csv')
ternopil_medium_1 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_22.csv')
ternopil_medium_2 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_14.csv')
ternopil_medium_3 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_23.csv')
ternopil_medium_4 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_24.csv')
ternopil_medium_5 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_13.csv')
ternopil_groos = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_01.csv')
ternopil_groos_2 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_02.csv')
ternopil_groos_3 = pd.read_csv('synthetic_data_ternopil_dubno/ternopil/ternopil_day_15.csv')

In [12]:
dubno_small = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_01.csv')
dubno_small_1 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_21.csv')
dubno_medium_1 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_22.csv')
dubno_medium_2 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_09.csv')
dubno_medium_3 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_17.csv')
dubno_medium_4 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_16.csv')
dubno_medium_5 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_14.csv')
dubno_gross = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_12.csv')
dubno_gross_2 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_11.csv')
dubno_gross_3 = pd.read_csv('synthetic_data_ternopil_dubno/dubno/dubno_day_07.csv')

In [13]:
def format_points(df):
    '''
    function to convert the data into Point class objects
    '''
    df = df.copy()
    df["point_id"] = df["point_id"].astype(str)
    points = []

    for _, row in df.iterrows():
        tw_start = str(row["tw_start"]).strip() if pd.notna(row["tw_start"]) else None
        tw_end = str(row["tw_end"]).strip() if pd.notna(row["tw_end"]) else None

        point_obj = Point(
            str(row["point_id"]),
            row["lan"],
            row["lot"],
            row["point_type"],
            tw_start,
            tw_end
        )

        points.append(point_obj)

    return points

In [14]:
kyiv_s_p = format_points(kyiv_small)
kyiv_s_1_p = format_points(kyiv_small_1)
kyiv_m_1_p = format_points(kyiv_meduim_1)
kyiv_m_2_p = format_points(kyiv_meduim_2)
kyiv_m_3_p = format_points(kyiv_meduim_3)
kyiv_m_4_p = format_points(kyiv_meduim_4)
kyiv_m_5_p = format_points(kyiv_meduim_5)
kyiv_g_1_p = format_points(kyiv_gross_1)
kyiv_g_2_p = format_points(kyiv_gross_2)
kyiv_g_3_p = format_points(kyiv_gross_3)

In [15]:
varash_s_p = format_points(varash_small)
varash_s_1_p = format_points(varash_small_1)
varash_m_1_p = format_points(varash_medium_1)
varash_m_2_p = format_points(varash_medium_2)
varash_m_3_p = format_points(varash_medium_3)
varash_m_4_p = format_points(varash_medium_4)
varash_m_5_p = format_points(varash_medium_5)
varash_g_1_p = format_points(varash_gross_1)
varash_g_2_p = format_points(varash_gross_2)
varash_g_3_p = format_points(varash_gross_3)

In [16]:
ternopil_s_p = format_points(ternopil_small)
ternopil_s_1_p = format_points(ternopil_small_1)
ternopil_m_1_p = format_points(ternopil_medium_1)
ternopil_m_2_p = format_points(ternopil_medium_2)
ternopil_m_3_p = format_points(ternopil_medium_3)
ternopil_m_4_p = format_points(ternopil_medium_4)
ternopil_m_5_p = format_points(ternopil_medium_5)
ternopil_g_p= format_points(ternopil_groos)
ternopil_g_2_p = format_points(ternopil_groos_2)
ternopil_g_3_p= format_points(ternopil_groos_3)

In [17]:
dubno_s_p = format_points(dubno_small)
dubno_s_1_p = format_points(dubno_small_1)
dubno_m_1_p = format_points(dubno_medium_1)
dubno_m_2_p = format_points(dubno_medium_2)
dubno_m_3_p = format_points(dubno_medium_3)
dubno_m_4_p = format_points(dubno_medium_4)
dubno_m_5_p = format_points(dubno_medium_5)
dubno_g_p = format_points(dubno_gross)
dubno_g_2_p = format_points(dubno_gross_2)
dubno_g_3_p = format_points(dubno_gross_3)

#### Define the params for points

In [18]:
def set_params(crew, crew_members, max_hours_work):
    result = dict()
    result['max_crews'] = crew
    result['max_workers'] =  crew*crew_members
    result['workers_per_crew'] =  crew_members
    result['max_route_duration_min'] = max_hours_work * 60
    return result

In [19]:
params_varash = set_params(varash['fixed_crew'], CREW_MEMBERS, MAX_WORK_HOURS)
params_kyiv = set_params(kyiv['fixed_crew'], CREW_MEMBERS, MAX_WORK_HOURS)

In [20]:
params_ternopil = set_params(ternopil['fixed_crew'], CREW_MEMBERS, MAX_WORK_HOURS)
params_dubno = set_params(dubno['fixed_crew'], CREW_MEMBERS, MAX_WORK_HOURS)

## Test clusterization preprocessing

In [21]:
import greedy_auto_time_start_1 as auto_alg
import greedy_time_simulate_1 as sim_alg

In [22]:
from aglo_klasters_2 import find_optimal_n_clusters
from build_matrices import build_matrices

In [23]:
cases = [
        {
            "name": "Kyiv_small",
            "depo": kyiv_depot,
            "stops": kyiv_s_p,
            "params": params_kyiv
        },
        {
            "name": "Kyiv_small_1",
            "depo": kyiv_depot,
            "stops": kyiv_s_1_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_medium_1",
            "depo": kyiv_depot,
            "stops": kyiv_m_1_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_medium_2",
            "depo": kyiv_depot,
            "stops": kyiv_m_2_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_medium_3",
            "depo": kyiv_depot,
            "stops": kyiv_m_3_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_medium_4",
            "depo": kyiv_depot,
            "stops": kyiv_m_4_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_medium_5",
            "depo": kyiv_depot,
            "stops": kyiv_m_5_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_gross_1",
            "depo": kyiv_depot,
            "stops": kyiv_g_1_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_gross_2",
            "depo": kyiv_depot,
            "stops": kyiv_g_2_p,
            "params": params_kyiv
        },
        {
           "name": "Kyiv_gross_3",
            "depo": kyiv_depot,
            "stops": kyiv_g_3_p,
            "params": params_kyiv
        },
        {
            "name": "Varash_small",
            "depo": varash_depot,
            "stops": varash_s_p,
            "params": params_varash
        },
        {
            "name": "Varash_small_1",
            "depo": varash_depot,
            "stops": varash_s_1_p,
            "params": params_varash
        },
        {
            "name": "Varash_medium_1",
            "depo": varash_depot,
            "stops": varash_m_1_p,
            "params": params_varash
        },
        {
            "name": "Varash_medium_2",
            "depo": varash_depot,
            "stops": varash_m_2_p,
            "params": params_varash
        },
        {
            "name": "Varash_medium_3",
            "depo": varash_depot,
            "stops": varash_m_3_p,
            "params": params_varash
        },
        {
            "name": "Varash_medium_4",
            "depo": varash_depot,
            "stops": varash_m_4_p,
            "params": params_varash
        },
        {
            "name": "Varash_medium_5",
            "depo": varash_depot,
            "stops": varash_m_5_p,
            "params": params_varash
        },
        {
            "name": "Varash_gross_1",
            "depo": varash_depot,
            "stops": varash_g_1_p,
            "params": params_varash
        },
        {
            "name": "Varash_gross_2",
            "depo": varash_depot,
            "stops": varash_g_2_p,
            "params": params_varash
        },
        {
            "name": "Varash_gross_3",
            "depo": varash_depot,
            "stops": varash_g_3_p,
            "params": params_varash
        }
]

In [24]:
cases_simple = [
        {
            "name": "ternopil_small",
            "depo": ternopil_depot,
            "stops": ternopil_s_p,
            "params": params_ternopil
        },
        {
            "name": "ternopil_small_1",
            "depo": ternopil_depot,
            "stops": ternopil_s_1_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_medium_1",
            "depo": ternopil_depot,
            "stops": ternopil_m_1_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_medium_2",
            "depo": ternopil_depot,
            "stops": ternopil_m_2_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_medium_3",
            "depo": ternopil_depot,
            "stops": ternopil_m_3_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_medium_4",
            "depo": ternopil_depot,
            "stops": ternopil_m_4_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_medium_5",
            "depo": ternopil_depot,
            "stops": ternopil_m_5_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_gross_1",
            "depo": ternopil_depot,
            "stops": ternopil_g_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_gross_2",
            "depo": ternopil_depot,
            "stops": ternopil_g_2_p,
            "params": params_ternopil
        },
        {
           "name": "ternopil_gross_3",
            "depo": ternopil_depot,
            "stops": ternopil_g_3_p,
            "params": params_ternopil
        },
        {
            "name": "dubno_small",
            "depo": dubno_depot,
            "stops": dubno_s_p,
            "params": params_dubno
        },
        {
            "name": "dubno_small_1",
            "depo": dubno_depot,
            "stops": dubno_s_1_p,
            "params": params_dubno
        },
        {
            "name": "dubno_medium_1",
            "depo": dubno_depot,
            "stops": dubno_m_1_p,
            "params": params_dubno
        },
        {
            "name": "dubno_medium_2",
            "depo": dubno_depot,
            "stops": dubno_m_2_p,
            "params": params_dubno
        },
        {
            "name": "dubno_medium_3",
            "depo": dubno_depot,
            "stops": dubno_m_3_p,
            "params": params_dubno
        },
        {
            "name": "dubno_medium_4",
            "depo": dubno_depot,
            "stops": dubno_m_4_p,
            "params": params_dubno
        },
        {
            "name": "dubno_medium_5",
            "depo": dubno_depot,
            "stops": dubno_m_5_p,
            "params": params_dubno
        },
        {
            "name": "dubno_gross_1",
            "depo": dubno_depot,
            "stops": dubno_g_p,
            "params": params_dubno
        },
        {
            "name": "dubno_gross_2",
            "depo": dubno_depot,
            "stops": dubno_g_2_p,
            "params": params_dubno
        },
        {
            "name": "dubno_gross_3",
            "depo": dubno_depot,
            "stops": dubno_g_3_p,
            "params": params_dubno
        }
]

## Aglomerative clusterization

### Additional functions for testing

In [25]:
def run_agg_clusterization(
    dataset,
    dist_km,
    time_min,
    build_routes_ag,
    linkage_mode,
    extra_kwargs=None):
    '''
    Run the agglomerative clasterization
    '''

    if extra_kwargs is None:
        extra_kwargs = {}

    params = dataset["params"]

    build_routes_kwargs = {
        "max_crews": params["max_crews"],
        "max_workers": params["max_workers"],
        "workers_per_crew": params["workers_per_crew"],
        "max_route_duration_min": params["max_route_duration_min"],
        **extra_kwargs
    }

    result = find_optimal_n_clusters(
        day_stops=dataset["stops"],
        depo=dataset["depo"],
        dist_km=dist_km,
        time_min=time_min,
        max_crews=params["max_crews"],
        build_routes_ag=build_routes_ag,
        build_routes_kwargs=build_routes_kwargs,
        linkage_mode=linkage_mode
    )

    labels = np.asarray(result["labels"])
    matrix = np.asarray(result["cluster_distance_matrix"], dtype=float)

    n_points = len(labels)
    n_clusters = len(np.unique(labels))

    if n_points < 2 or n_clusters < 2 or n_clusters >= n_points:
        result["silhouette_score"] = np.nan
    else:
        result["silhouette_score"] = silhouette_score(
            matrix,
            labels,
            metric="precomputed"
        )

    return result

In [26]:
def test_linkage_methods_for_case(
    case,
    dist_km,
    time_min,
    build_routes_ag,
    linkage_modes,
    extra_kwargs=None):
    '''
    fucntion to run clustering using input linkage and one of aproaches of greedy
    '''

    rows = []
    raw_results = {}


    for linkage_mode in linkage_modes:
        try:
            result = run_agg_clusterization(
                dataset=case,
                dist_km=dist_km,
                time_min=time_min,
                build_routes_ag=build_routes_ag,
                linkage_mode=linkage_mode,
                extra_kwargs=extra_kwargs,
            )

            summary = result["summary"]

            row = {
                "Dataset": case["name"],
                "linkage": linkage_mode,
                "status": "ok",
                "n_clusters": len(set(result["labels"])),
                "n_routes": summary["n_routes"],
                "total_distance_km": summary["total_distance_km"],
                "total_duration_min": summary["total_duration_min"],
                "total_served": summary["total_served"],
                "silhouette_score": result["silhouette_score"],
            }

            rows.append(row)
            raw_results[linkage_mode] = result

        except Exception as e:
            print(f"[case={case['name']}] [linkage={linkage_mode}] error: {e}")
            rows.append({
                "Dataset": case["name"],
                "linkage": linkage_mode,
                "status": f"error: {e}",
                "n_clusters": None,
                "n_routes": None,
                "total_distance_km": None,
                "total_duration_min": None,
                "total_served": None,
                "silhouette_score": None,
            })
            raw_results[linkage_mode] = None

    df_case = pd.DataFrame(rows)

    no_troubles = df_case[df_case["status"] == "ok"].copy()

    best_linkage = None
    if not no_troubles.empty:
        no_troubles = no_troubles.sort_values(
            by=["total_duration_min", "total_distance_km"],
            ascending=[True, True]
        )
        best_linkage = no_troubles.iloc[0]["linkage"]

    df_case["is_best"] = df_case["linkage"].eq(best_linkage)

    return df_case, raw_results, best_linkage

In [27]:
def run_clusters_linkage_experiment(
    cases,
    linkage_modes,
    greedy_type,
    method_of_greedy_name,
    extra_kwargs=None):
    '''
    fucntion to choose the best linkage for agglomerative using 2 aproaches of greedy
    '''
    all_rows = []
    best_results = {}

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"

        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=BLOCK_SIZE,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        df_case, raw_results, best_linkage = test_linkage_methods_for_case(
            case=case,
            dist_km=dist_km,
            time_min=time_min,
            build_routes_ag=greedy_type,
            linkage_modes=linkage_modes,
            extra_kwargs=extra_kwargs
        )

        
        df_case["Method"] = method_of_greedy_name
        all_rows.append(df_case)

        if best_linkage is not None:
            best_results[name] = {
                "method": method_of_greedy_name,
                "best_linkage": best_linkage,
                "best_result": raw_results[best_linkage],
            }
        else:
            best_results[name] = {
                "method": method_of_greedy_name,
                "best_linkage": None,
                "best_result": None,
            }

    final_df = pd.concat(all_rows, ignore_index=True)

    return final_df, best_results

### Test the linkage

In [29]:
linkage_modes_to_test=["average", "complete", "single"]

In [30]:
df_math, best_math = run_clusters_linkage_experiment(
    cases=cases,
    linkage_modes=linkage_modes_to_test,
    greedy_type=auto_alg.build_routes,
    method_of_greedy_name="math_start")

In [31]:
df_sim, best_sim = run_clusters_linkage_experiment(
    cases=cases,
    linkage_modes=linkage_modes_to_test,
    greedy_type=sim_alg.build_routes,
    method_of_greedy_name="simulation_start",
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [32]:
df_math_simple, best_math_simple = run_clusters_linkage_experiment(
    cases=cases_simple,
    linkage_modes=linkage_modes_to_test,
    greedy_type=auto_alg.build_routes,
    method_of_greedy_name="math_start")

In [33]:
df_sim_simple, best_sim_simple = run_clusters_linkage_experiment(
    cases=cases_simple,
    linkage_modes=linkage_modes_to_test,
    greedy_type=sim_alg.build_routes,
    method_of_greedy_name="simulation_start",
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

### Results of the linkage experiment

In [34]:
df_all = pd.concat(
    [df_math, df_sim, df_math_simple, df_sim_simple],
    ignore_index=True
)

In [35]:
summary_robust = (
    df_all.groupby("linkage")
    .agg(
        n_best=("is_best", "sum"),
        avg_duration=("total_duration_min", "mean"),
        median_duration=("total_duration_min", "median"),
        avg_distance=("total_distance_km", "mean"),
        median_distance=("total_distance_km", "median"),
        avg_routes=("n_routes", "mean"),
        median_routes=("n_routes", "median"),
        avg_silhouette=("silhouette_score", "mean"),
        median_silhouette=("silhouette_score", "median"),
        n_silhouette_defined=("silhouette_score", lambda x: x.notna().sum())
    )
    .reset_index()
    .sort_values(
        by=["n_best", "avg_duration", "avg_distance"],
        ascending=[False, True, True]
    )
)

summary_robust

,linkage,n_best,avg_duration,median_duration,avg_distance,median_distance,avg_routes,median_routes,avg_silhouette,median_silhouette,n_silhouette_defined
1,complete,39,4181.189188,1676.850833,945.429530,459.31345,6.750,2.5,0.370910,0.349098,47
0,average,39,4336.659208,1676.850833,970.379540,476.20625,6.975,2.5,0.406144,0.366634,32
2,single,2,4467.387625,1676.850833,1005.157369,491.40425,6.900,2.5,0.314234,0.296527,11


In [36]:
def linkage_block_summary(df, block_name):
    df_ok = df[df["status"] == "ok"].copy()

    summary = (
        df_ok.groupby("linkage")
        .agg(
            n_cases=("Dataset", "count"),
            n_best=("is_best", "sum"),
            avg_duration=("total_duration_min", "mean"),
            median_duration=("total_duration_min", "median"),
            avg_distance=("total_distance_km", "mean"),
            median_distance=("total_distance_km", "median"),
            avg_routes=("n_routes", "mean"),
            avg_clusters=("n_clusters", "mean"),
            avg_silhouette=("silhouette_score", "mean"),
            median_silhouette=("silhouette_score", "median"),
            n_silhouette_defined=("silhouette_score", lambda x: x.notna().sum())
        )
        .reset_index()
        .sort_values(
            by=["n_best", "avg_duration", "avg_distance"],
            ascending=[False, True, True]
        )
    )

    summary.insert(0, "block", block_name)
    return summary

In [37]:
summary_math = linkage_block_summary(df_math, "dataset1_math")
summary_sim = linkage_block_summary(df_sim, "dataset1_simulation")
summary_math_simple = linkage_block_summary(df_math_simple, "dataset2_math")
summary_sim_simple = linkage_block_summary(df_sim_simple, "dataset2_simulation")

In [38]:
print("------ Dataset Kyiv-Varash + math ------")
display(summary_math)

print("------ Dataset Kyiv-Varash + simulation ------")
display(summary_sim)

print("------ Dataset Ternopil-Dubno + math ------")
display(summary_math_simple)

print("------ Dataset Ternopil-Dubno + simulation ------")
display(summary_sim_simple)

------ Dataset Kyiv-Varash + math ------


,block,linkage,n_cases,n_best,avg_duration,median_duration,avg_distance,median_distance,avg_routes,avg_clusters,avg_silhouette,median_silhouette,n_silhouette_defined
1,dataset1_math,complete,20,10,6663.386833,5846.536667,1602.80545,1405.07555,10.05,3.65,0.392876,0.366734,13
0,dataset1_math,average,20,8,6783.212500,5720.495000,1646.42333,1469.03675,10.05,2.15,0.421199,0.381471,10
2,dataset1_math,single,20,2,6869.211667,5748.745000,1692.69824,1564.82325,10.05,1.15,0.337802,0.337802,2


------ Dataset Kyiv-Varash + simulation ------


,block,linkage,n_cases,n_best,avg_duration,median_duration,avg_distance,median_distance,avg_routes,avg_clusters,avg_silhouette,median_silhouette,n_silhouette_defined
1,dataset1_simulation,complete,20,13,5654.343750,4848.945833,1522.877515,1436.92820,9.40,4.05,0.404538,0.374253,14
0,dataset1_simulation,average,20,7,5950.242917,5045.112500,1560.620140,1460.31975,10.05,4.30,0.410704,0.375258,12
2,dataset1_simulation,single,20,0,6323.951917,5403.530000,1643.513710,1538.36805,9.90,1.90,0.273879,0.207434,5


------ Dataset Ternopil-Dubno + math ------


,block,linkage,n_cases,n_best,avg_duration,median_duration,avg_distance,median_distance,avg_routes,avg_clusters,avg_silhouette,median_silhouette,n_silhouette_defined
0,dataset2_math,average,20,12,2436.769333,1854.190000,338.774355,266.2330,3.95,1.45,0.367197,0.336793,4
1,dataset2_math,complete,20,8,2303.372417,1834.150833,327.686650,264.5892,3.80,1.85,0.328375,0.290723,9
2,dataset2_math,single,20,0,2448.310000,1854.190000,345.942410,266.2330,3.85,1.00,NaN,NaN,0


------ Dataset Ternopil-Dubno + simulation ------


,block,linkage,n_cases,n_best,avg_duration,median_duration,avg_distance,median_distance,avg_routes,avg_clusters,avg_silhouette,median_silhouette,n_silhouette_defined
0,dataset2_simulation,average,20,12,2176.412083,1526.845833,335.700335,263.33980,3.85,1.65,0.397898,0.370700,6
1,dataset2_simulation,complete,20,8,2103.653750,1476.240833,328.348505,262.67775,3.75,2.00,0.336951,0.345256,11
2,dataset2_simulation,single,20,0,2228.076917,1526.845833,338.475115,263.33980,3.80,1.35,0.352892,0.406898,4


### The best result Agglomerative

In [28]:
linkage_mode_the_best = ['complete']

In [29]:
def run_clasters_agg(cases, linkage_mode_the_best):
    results = []
    target_duration = 480

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"
        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=25,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        df_auto, raw_auto, best_auto = test_linkage_methods_for_case(
            case=case,
            dist_km=dist_km,
            time_min=time_min,
            build_routes_ag=auto_alg.build_routes,
            linkage_modes=linkage_mode_the_best
        )

        df_sim, raw_sim, best_sim = test_linkage_methods_for_case(
            case=case,
            dist_km=dist_km,
            time_min=time_min,
            build_routes_ag=sim_alg.build_routes,
            linkage_modes=linkage_mode_the_best,
            extra_kwargs={
                "start_from": "08:00",
                "start_to": "10:00",
                "step_min": 10,
                "target_duration": target_duration
            }
        )


        result_auto = raw_auto.get(linkage_mode_the_best[0])
        result_sim = raw_sim.get(linkage_mode_the_best[0])

        results.append({
            "Dataset": name,
            "Module": "Agglo_Math_Start",
            "linkage": linkage_mode_the_best,
            "n_clusters": len(set(result_auto["labels"])) if result_auto is not None else None,
            "n_routes": result_auto["summary"]["n_routes"] if result_auto is not None else None,
            "total_distance_km": result_auto["summary"]["total_distance_km"] if result_auto is not None else None,
            "total_duration_min": result_auto["summary"]["total_duration_min"] if result_auto is not None else None,
            "total_served": result_auto["summary"]["total_served"] if result_auto is not None else None,
            "silhouette_score": result_auto["silhouette_score"] if result_auto is not None else None
        })

        results.append({
            "Dataset": name,
            "Module": "Agglo_Simulate",
            "linkage": linkage_mode_the_best,
            "n_clusters": len(set(result_sim["labels"])) if result_sim is not None else None,
            "n_routes": result_sim["summary"]["n_routes"] if result_sim is not None else None,
            "total_distance_km": result_sim["summary"]["total_distance_km"] if result_sim is not None else None,
            "total_duration_min": result_sim["summary"]["total_duration_min"] if result_sim is not None else None,
            "total_served": result_sim["summary"]["total_served"] if result_sim is not None else None,
            "silhouette_score": result_sim["silhouette_score"] if result_sim is not None else None
        })

    return pd.DataFrame(results)

In [ ]:
the_best_ag_res_original = run_clasters_agg(cases, linkage_mode_the_best)

In [259]:
the_best_ag_res_simple = run_clasters_agg(cases_simple, linkage_mode_the_best)

## K-medoids

### Test k-medoids preprocessing

In [187]:
from k_medoids_2 import find_optimal_n_clusters_kmedoids

In [188]:
def run_kmedoids_clusterization(
    dataset,
    dist_km,
    time_min,
    build_routes_fn,
    random_state=42,
    max_iter=100,
    extra_kwargs=None):
    '''
    clasterization for one day for one algorithm
    '''

    if extra_kwargs is None:
        extra_kwargs = {}

    params = dataset["params"]

    build_routes_kwargs = {
        "max_crews": params["max_crews"],
        "max_workers": params["max_workers"],
        "workers_per_crew": params["workers_per_crew"],
        "max_route_duration_min": params["max_route_duration_min"],
        **extra_kwargs
    }

    result = find_optimal_n_clusters_kmedoids(
        day_stops=dataset["stops"],
        depo=dataset["depo"],
        dist_km=dist_km,
        time_min=time_min,
        max_crews=params["max_crews"],
        build_routes_ag=build_routes_fn,
        build_routes_kwargs=build_routes_kwargs,
        random_state=random_state,
        max_iter=max_iter
    )

    labels = np.asarray(result["labels"])
    matrix = np.asarray(result["cluster_distance_matrix"], dtype=float)

    n_points = len(labels)
    n_clusters_final = len(np.unique(labels))

    if n_points < 2 or n_clusters_final < 2 or n_clusters_final >= n_points:
        result["silhouette_score"] = np.nan
    else:
        result["silhouette_score"] = silhouette_score(
            matrix,
            labels,
            metric="precomputed"
        )

    result["n_clusters_final"] = n_clusters_final
    return result

In [189]:
def run_kmedoids_iterations_experiment(
    cases,
    build_routes_fn,
    method_name,
    max_iter_values,
    random_state,
    extra_kwargs=None):
    '''
    run different iterations with fixed seed
    '''

    rows = []

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"

        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=BLOCK_SIZE,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        for max_iter in max_iter_values:
            try:
                result = run_kmedoids_clusterization(
                    dataset=case,
                    dist_km=dist_km,
                    time_min=time_min,
                    build_routes_fn=build_routes_fn,
                    random_state=random_state,
                    max_iter=max_iter,
                    extra_kwargs=extra_kwargs
                )

                summary = result["summary"]

                rows.append({
                    "Dataset": case["name"],
                    "Method": method_name,
                    "max_iter": max_iter,
                    "random_state": random_state,
                    "status": "ok",
                    "n_clusters_final": result["n_clusters_final"],
                    "n_routes": summary["n_routes"],
                    "total_distance_km": summary["total_distance_km"],
                    "total_duration_min": summary["total_duration_min"],
                    "total_served": summary["total_served"],
                    "silhouette_score": result["silhouette_score"],
                    "final_iterations": result["num_iterations"],
                    "cost": result["costs"],
                    "result_obj": result
                })

            except Exception as e:
                rows.append({
                    "Dataset": case["name"],
                    "Method": method_name,
                    "max_iter": max_iter,
                    "random_state": random_state,
                    "status": f"fail: {e}",
                    "n_clusters_final": np.nan,
                    "n_routes": np.nan,
                    "total_distance_km": np.nan,
                    "total_duration_min": np.nan,
                    "total_served": np.nan,
                    "silhouette_score": np.nan,
                    "final_iterations": np.nan,
                    "cost": np.nan,
                    "result_obj": None
                })

    return pd.DataFrame(rows)

In [190]:
def add_iteration_improvements_vs_baseline(df, baseline_iter=1):
    '''
    comparison with baseline
    '''
    df = df.copy()

    baseline = (
        df[df["max_iter"] == baseline_iter][[
            "Dataset",
            "Method",
            "total_duration_min",
            "total_distance_km"
        ]]
        .rename(columns={
            "total_duration_min": "baseline_duration",
            "total_distance_km": "baseline_distance"
        })
    )

    df = df.merge(baseline, on=["Dataset", "Method"], how="left")

    df["duration_diff_abs"] = df["baseline_duration"] - df["total_duration_min"]
    df["distance_diff_abs"] = df["baseline_distance"] - df["total_distance_km"]

    df["duration_improvement_pct"] = np.where(
        df["baseline_duration"] > 0,
        df["duration_diff_abs"] / df["baseline_duration"] * 100,
        np.nan
    )

    df["distance_improvement_pct"] = np.where(
        df["baseline_distance"] > 0,
        df["distance_diff_abs"] / df["baseline_distance"] * 100,
        np.nan
    )

    return df

In [191]:
def summarize_iteration_experiment(df, group_col=None):
    '''
    summary for each dataset
    '''
    df_ok = df[df["status"] == "ok"].copy()

    grouping_cols = ["Method", "max_iter"]
    if group_col is not None:
        grouping_cols.insert(0, group_col)

    summary = (
        df_ok.groupby(grouping_cols)
        .agg(
            n_cases=("Dataset", "count"),
            avg_clusters=("n_clusters_final", "mean"),
            avg_duration=("total_duration_min", "mean"),
            avg_distance=("total_distance_km", "mean"),
            avg_duration_improvement_pct=("duration_improvement_pct", "mean"),
            avg_distance_improvement_pct=("distance_improvement_pct", "mean"),
            avg_silhouette=("silhouette_score", "mean"),
            avg_final_iterations=("final_iterations", "mean")
        )
        .reset_index()
        .sort_values(grouping_cols)
    )

    return summary

### Test max number of iters for k-medoids

In [194]:
max_iter_values = [1, 5, 10, 20, 50, 100, 300, 500]
fixed_random_state = 42

In [195]:
df_iter_math = run_kmedoids_iterations_experiment(
    cases=cases,
    build_routes_fn=auto_alg.build_routes,
    method_name="math_start",
    max_iter_values=max_iter_values,
    random_state=fixed_random_state
)

In [196]:
df_iter_sim = run_kmedoids_iterations_experiment(
    cases=cases,
    build_routes_fn=sim_alg.build_routes,
    method_name="simulation_start",
    max_iter_values=max_iter_values,
    random_state=fixed_random_state,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [197]:
df_simple_iter_math = run_kmedoids_iterations_experiment(
    cases=cases_simple,
    build_routes_fn=auto_alg.build_routes,
    method_name="math_start",
    max_iter_values=max_iter_values,
    random_state=fixed_random_state
)

In [198]:
df_simple_iter_sim = run_kmedoids_iterations_experiment(
    cases=cases_simple,
    build_routes_fn=sim_alg.build_routes,
    method_name="simulation_start",
    max_iter_values=max_iter_values,
    random_state=fixed_random_state,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [199]:
experiment_dfs = {
    "Dataset Kyiv-Varash + math": df_iter_math,
    "Dataset Kyiv-Varash + simulation": df_iter_sim,
    "Dataset Ternopil-Dubno + math": df_simple_iter_math,
    "Dataset Ternopil-Dubno + simulation": df_simple_iter_sim,
}

In [200]:
all_detailed = {}
all_summary = {}

for exp_name, df in experiment_dfs.items():
    detailed = add_iteration_improvements_vs_baseline(df, baseline_iter=1)
    summary = summarize_iteration_experiment(detailed)

    all_detailed[exp_name] = detailed
    all_summary[exp_name] = summary
    print(f"\n---------- {exp_name} ----------")
    display(
        summary[
            [
                "Method",
                "max_iter",
                "n_cases",
                "avg_duration",
                "avg_distance",
                "avg_duration_improvement_pct",
                "avg_distance_improvement_pct",
                "avg_final_iterations",
                "avg_silhouette"
            ]
        ].sort_values(["Method", "max_iter"])
    )


---------- Dataset Kyiv-Varash + math ----------


,Method,max_iter,n_cases,avg_duration,avg_distance,avg_duration_improvement_pct,avg_distance_improvement_pct,avg_final_iterations,avg_silhouette
0,math_start,1,20,6636.150583,1601.311145,0.000000,0.000000,1.0,0.399946
1,math_start,5,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
2,math_start,10,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
3,math_start,20,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
4,math_start,50,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
5,math_start,100,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
6,math_start,300,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423
7,math_start,500,20,6504.993167,1585.280825,1.105204,0.552628,3.1,0.400423



---------- Dataset Kyiv-Varash + simulation ----------


,Method,max_iter,n_cases,avg_duration,avg_distance,avg_duration_improvement_pct,avg_distance_improvement_pct,avg_final_iterations,avg_silhouette
0,simulation_start,1,20,5485.204167,1508.01674,0.000000,0.000000,1.00,0.390759
1,simulation_start,5,20,5357.695417,1505.89875,1.327521,0.149316,3.45,0.407745
2,simulation_start,10,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321
3,simulation_start,20,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321
4,simulation_start,50,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321
5,simulation_start,100,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321
6,simulation_start,300,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321
7,simulation_start,500,20,5289.987250,1497.18005,1.965708,0.456629,3.90,0.406321



---------- Dataset Ternopil-Dubno + math ----------


,Method,max_iter,n_cases,avg_duration,avg_distance,avg_duration_improvement_pct,avg_distance_improvement_pct,avg_final_iterations,avg_silhouette
0,math_start,1,20,2360.93975,337.391145,0.000000,0.000000,1.0,0.293972
1,math_start,5,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
2,math_start,10,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
3,math_start,20,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
4,math_start,50,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
5,math_start,100,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
6,math_start,300,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672
7,math_start,500,20,2356.82750,335.633195,0.105579,0.252888,2.2,0.302672



---------- Dataset Ternopil-Dubno + simulation ----------


,Method,max_iter,n_cases,avg_duration,avg_distance,avg_duration_improvement_pct,avg_distance_improvement_pct,avg_final_iterations,avg_silhouette
0,simulation_start,1,20,2133.083583,334.378090,0.000000,0.000000,1.0,0.259140
1,simulation_start,5,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
2,simulation_start,10,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
3,simulation_start,20,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
4,simulation_start,50,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
5,simulation_start,100,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
6,simulation_start,300,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602
7,simulation_start,500,20,2129.928417,329.950185,0.126113,0.813763,2.5,0.291602


In [201]:
def find_recommended_max_iter(summary_df, improvement_threshold=0.5):
    '''
    find the best num of iterations for each dataset
    '''
    recommendations = []

    for method in summary_df["Method"].dropna().unique():
        sub = summary_df[summary_df["Method"] == method].sort_values("max_iter").copy()

        candidate = sub[
            (sub["avg_duration_improvement_pct"] >= 0) &
            (sub["avg_distance_improvement_pct"] >= 0)
        ].copy()

        recommended_iter = None

        for _, row in candidate.iterrows():
            duration_gain = row["avg_duration_improvement_pct"]
            distance_gain = row["avg_distance_improvement_pct"]

            if duration_gain >= 0 and distance_gain >= 0:
                if duration_gain <= improvement_threshold and distance_gain <= improvement_threshold:
                    recommended_iter = row["max_iter"]
                    break

        if recommended_iter is None:
            recommended_iter = sub.iloc[-1]["max_iter"]

        best_distance_row = sub.loc[sub["avg_distance_improvement_pct"].idxmax()]
        best_duration_row = sub.loc[sub["avg_duration_improvement_pct"].idxmax()]

        recommendations.append({
            "Method": method,
            "recommended_max_iter": recommended_iter,
            "best_iter_by_distance": best_distance_row["max_iter"],
            "best_distance_improvement_pct": best_distance_row["avg_distance_improvement_pct"],
            "best_iter_by_duration": best_duration_row["max_iter"],
            "best_duration_improvement_pct": best_duration_row["avg_duration_improvement_pct"],
        })

    return pd.DataFrame(recommendations)

In [204]:
recommended_tables = {}

for exp_name, summary in all_summary.items():
    rec = find_recommended_max_iter(summary, improvement_threshold=0.1)
    recommended_tables[exp_name] = rec

    print(f"\n---------- RECOMMENDED MAX_ITER: {exp_name} ----------")
    display(rec)


---------- RECOMMENDED MAX_ITER: Dataset Kyiv-Varash + math ----------


,Method,recommended_max_iter,best_iter_by_distance,best_distance_improvement_pct,best_iter_by_duration,best_duration_improvement_pct
0,math_start,1,5,0.552628,5,1.105204



---------- RECOMMENDED MAX_ITER: Dataset Kyiv-Varash + simulation ----------


,Method,recommended_max_iter,best_iter_by_distance,best_distance_improvement_pct,best_iter_by_duration,best_duration_improvement_pct
0,simulation_start,1,10,0.456629,10,1.965708



---------- RECOMMENDED MAX_ITER: Dataset Ternopil-Dubno + math ----------


,Method,recommended_max_iter,best_iter_by_distance,best_distance_improvement_pct,best_iter_by_duration,best_duration_improvement_pct
0,math_start,1,5,0.252888,5,0.105579



---------- RECOMMENDED MAX_ITER: Dataset Ternopil-Dubno + simulation ----------


,Method,recommended_max_iter,best_iter_by_distance,best_distance_improvement_pct,best_iter_by_duration,best_duration_improvement_pct
0,simulation_start,1,5,0.813763,5,0.126113


### Test seed for K-medoids

In [205]:
seed_values = [1, 5, 10, 20, 42, 100]
best_iter_num = 10

In [206]:
def run_kmedoids_seed_experiment(
    cases,
    build_routes_fn,
    method_name,
    random_state_values,
    max_iter,
    extra_kwargs=None):
    '''
    Runs k-medoids for different random_state values with fixed max_iter
    '''

    rows = []

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"

        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=BLOCK_SIZE,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        for random_state in random_state_values:
            try:
                result = run_kmedoids_clusterization(
                    dataset=case,
                    dist_km=dist_km,
                    time_min=time_min,
                    build_routes_fn=build_routes_fn,
                    random_state=random_state,
                    max_iter=max_iter,
                    extra_kwargs=extra_kwargs
                )

                summary = result["summary"]

                rows.append({
                    "Dataset": case["name"],
                    "Method": method_name,
                    "random_state": random_state,
                    "max_iter": max_iter,
                    "status": "ok",
                    "n_clusters_final": result["n_clusters_final"],
                    "n_routes": summary["n_routes"],
                    "total_distance_km": summary["total_distance_km"],
                    "total_duration_min": summary["total_duration_min"],
                    "total_served": summary["total_served"],
                    "silhouette_score": result.get("silhouette_score", np.nan),
                    "final_iterations": result.get("num_iterations", np.nan)
                })

            except Exception as e:
                rows.append({
                    "Dataset": case["name"],
                    "Method": method_name,
                    "random_state": random_state,
                    "max_iter": max_iter,
                    "status": f"error: {e}",
                    "n_clusters_final": np.nan,
                    "n_routes": np.nan,
                    "total_distance_km": np.nan,
                    "total_duration_min": np.nan,
                    "total_served": np.nan,
                    "silhouette_score": np.nan,
                    "final_iterations": np.nan
                })

    return pd.DataFrame(rows)

In [207]:
df_seed_math = run_kmedoids_seed_experiment(
    cases=cases,
    build_routes_fn=auto_alg.build_routes,
    method_name="math_start",
    random_state_values=seed_values,
    max_iter=best_iter_num
)

In [209]:
df_seed_sim = run_kmedoids_seed_experiment(
    cases=cases,
    build_routes_fn=sim_alg.build_routes,
    method_name="simulation_start",
    random_state_values=seed_values,
    max_iter=best_iter_num,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [210]:
df_simple_seed_math = run_kmedoids_seed_experiment(
    cases=cases_simple,
    build_routes_fn=auto_alg.build_routes,
    method_name="math_start",
    random_state_values=seed_values,
    max_iter=best_iter_num
)

In [211]:
df_simple_seed_sim = run_kmedoids_seed_experiment(
    cases=cases_simple,
    build_routes_fn=sim_alg.build_routes,
    method_name="simulation_start",
    random_state_values=seed_values,
    max_iter=best_iter_num,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [212]:
seed_experiment_dfs = {
    "Dataset Kyiv-Varash + math": df_seed_math,
    "Dataset Kyiv-Varash + simulation": df_seed_sim,
    "Dataset Ternopil-Dubno + math": df_simple_seed_math,
    "Dataset Ternopil-Dubno + simulation": df_simple_seed_sim,
}

In [213]:
def summarize_seed_experiment(df, group_col="Dataset"):
    '''
    Summary of random state influence on results
    '''
    df_ok = df[df["status"] == "ok"].copy()

    grouping_cols = [group_col, "Method"] if group_col else ["Method"]

    summary = (
        df_ok.groupby(grouping_cols)
        .agg(
            n_runs=("random_state", "count"),
            n_unique_seeds=("random_state", "nunique"),
            n_unique_clusters_found=("n_clusters_final", "nunique"),

            mean_duration=("total_duration_min", "mean"),
            std_duration=("total_duration_min", "std"),
            min_duration=("total_duration_min", "min"),
            max_duration=("total_duration_min", "max"),

            mean_distance=("total_distance_km", "mean"),
            std_distance=("total_distance_km", "std"),
            min_distance=("total_distance_km", "min"),
            max_distance=("total_distance_km", "max"),

            mean_silhouette=("silhouette_score", "mean"),
            std_silhouette=("silhouette_score", "std"),

            mean_final_iterations=("final_iterations", "mean"),
            std_final_iterations=("final_iterations", "std"),
        )
        .reset_index()
    )

    summary["duration_range_abs"] = summary["max_duration"] - summary["min_duration"]
    summary["distance_range_abs"] = summary["max_distance"] - summary["min_distance"]

    summary["duration_range_pct_vs_min"] = np.where(
        summary["min_duration"] > 0,
        summary["duration_range_abs"] / summary["min_duration"] * 100,
        np.nan
    )

    summary["distance_range_pct_vs_min"] = np.where(
        summary["min_distance"] > 0,
        summary["distance_range_abs"] / summary["min_distance"] * 100,
        np.nan
    )

    summary["duration_cv_pct"] = np.where(
        summary["mean_duration"] > 0,
        summary["std_duration"] / summary["mean_duration"] * 100,
        np.nan
    )

    summary["distance_cv_pct"] = np.where(
        summary["mean_distance"] > 0,
        summary["std_distance"] / summary["mean_distance"] * 100,
        np.nan
    )

    return summary.sort_values(grouping_cols)

In [214]:
all_seed_summary = {}

for exp_name, df in seed_experiment_dfs.items():
    summary = summarize_seed_experiment(df, group_col="Dataset")
    all_seed_summary[exp_name] = summary

In [215]:
def seed_summary(summary_df):
    if summary_df.empty:
        return pd.DataFrame()

    result = summary_df.copy()

    result["max_range_pct"] = result[
        ["duration_range_pct_vs_min", "distance_range_pct_vs_min"]
    ].max(axis=1)

    result["seed_effect"] = np.select(
        [
            (result["max_range_pct"] < 2) & (result["n_unique_clusters_found"] == 1),
            (result["max_range_pct"] < 5),
            (result["max_range_pct"] < 10),
            (result["max_range_pct"] >= 10),
        ],
        [
            "no noticeable effect",
            "small effect",
            "moderate effect",
            "strong effect",
        ],
        default="moderate effect"
    )

    cols = [
        "Dataset",
        "Method",
        "n_runs",
        "n_unique_clusters_found",
        "min_duration",
        "max_duration",
        "duration_range_pct_vs_min",
        "min_distance",
        "max_distance",
        "distance_range_pct_vs_min",
        "max_range_pct",
        "seed_effect"
    ]

    cols = [c for c in cols if c in result.columns]
    return result[cols].sort_values(["Dataset", "Method"])

In [216]:
for exp_name, summary in all_seed_summary.items():
    print(f"\n===== COMPACT SUMMARY: {exp_name} =====")
    display(seed_summary(summary))


===== COMPACT SUMMARY: Dataset Kyiv-Varash + math =====


,Dataset,Method,n_runs,n_unique_clusters_found,min_duration,max_duration,duration_range_pct_vs_min,min_distance,max_distance,distance_range_pct_vs_min,max_range_pct,seed_effect
0,Kyiv_gross_1,math_start,6,4,13935.291667,14290.873333,2.551663,2975.1242,3141.6338,5.596728,5.596728,moderate effect
1,Kyiv_gross_2,math_start,6,2,14067.520000,14291.818333,1.594441,3015.3239,3126.9540,3.702093,3.702093,small effect
2,Kyiv_gross_3,math_start,6,3,12926.500000,14142.095000,9.403899,2973.1037,3345.2930,12.518544,12.518544,strong effect
3,Kyiv_medium_1,math_start,6,2,10522.398333,11055.093333,5.062487,2743.6819,2866.5526,4.478314,5.062487,moderate effect
4,Kyiv_medium_2,math_start,6,4,12817.210000,13277.913333,3.594412,2680.9977,2917.0760,8.805614,8.805614,moderate effect
5,Kyiv_medium_3,math_start,6,3,10931.098333,11513.598333,5.328833,2492.2966,2544.5073,2.094883,5.328833,moderate effect
6,Kyiv_medium_4,math_start,6,5,13415.496667,14273.428333,6.395079,2944.2642,3110.5435,5.647567,6.395079,moderate effect
7,Kyiv_medium_5,math_start,6,4,9687.306667,10312.106667,6.449677,2365.8819,2543.9306,7.525680,7.525680,moderate effect
8,Kyiv_small,math_start,6,5,10455.823333,11913.560000,13.941864,2525.7054,2739.4515,8.462828,13.941864,strong effect
9,Kyiv_small_1,math_start,6,6,9882.310000,10552.010000,6.776756,2308.2417,2549.6792,10.459802,10.459802,strong effect



===== COMPACT SUMMARY: Dataset Kyiv-Varash + simulation =====


,Dataset,Method,n_runs,n_unique_clusters_found,min_duration,max_duration,duration_range_pct_vs_min,min_distance,max_distance,distance_range_pct_vs_min,max_range_pct,seed_effect
0,Kyiv_gross_1,simulation_start,6,4,11342.638333,11932.621667,5.201465,2657.3836,2928.6072,10.206415,10.206415,strong effect
1,Kyiv_gross_2,simulation_start,6,4,11101.686667,11818.701667,6.458613,2634.6635,2953.4625,12.100179,12.100179,strong effect
2,Kyiv_gross_3,simulation_start,6,4,10140.548333,10790.155000,6.406031,2731.3257,2829.0863,3.579236,6.406031,moderate effect
3,Kyiv_medium_1,simulation_start,6,5,9090.510000,9836.696667,8.208414,2572.9880,2855.2017,10.968326,10.968326,strong effect
4,Kyiv_medium_2,simulation_start,6,5,9381.165000,10595.470000,12.944075,2420.5223,2653.2506,9.614797,12.944075,strong effect
5,Kyiv_medium_3,simulation_start,6,3,8429.335000,8940.205000,6.060620,2496.2005,2561.1512,2.601982,6.060620,moderate effect
6,Kyiv_medium_4,simulation_start,6,4,10779.096667,11333.400000,5.142391,2719.5007,2847.7281,4.715108,5.142391,moderate effect
7,Kyiv_medium_5,simulation_start,6,4,7317.855000,7924.836667,8.294530,2150.8203,2318.1674,7.780617,8.294530,moderate effect
8,Kyiv_small,simulation_start,6,5,8634.493333,9146.285000,5.927292,2407.5615,2538.5042,5.438810,5.927292,moderate effect
9,Kyiv_small_1,simulation_start,6,5,7849.446667,8631.651667,9.965097,2225.3798,2395.8757,7.661429,9.965097,moderate effect



===== COMPACT SUMMARY: Dataset Ternopil-Dubno + math =====


,Dataset,Method,n_runs,n_unique_clusters_found,min_duration,max_duration,duration_range_pct_vs_min,min_distance,max_distance,distance_range_pct_vs_min,max_range_pct,seed_effect
0,dubno_gross_1,math_start,6,1,929.873333,929.873333,0.000000,143.8454,143.8454,0.000000,0.000000,no noticeable effect
1,dubno_gross_2,math_start,6,1,574.731667,574.731667,0.000000,121.0353,121.0353,0.000000,0.000000,no noticeable effect
2,dubno_gross_3,math_start,6,1,889.795000,889.795000,0.000000,140.5578,140.5578,0.000000,0.000000,no noticeable effect
3,dubno_medium_1,math_start,6,1,515.701667,515.701667,0.000000,109.8807,109.8807,0.000000,0.000000,no noticeable effect
4,dubno_medium_2,math_start,6,1,688.988333,688.988333,0.000000,126.4482,126.4482,0.000000,0.000000,no noticeable effect
5,dubno_medium_3,math_start,6,2,598.050000,691.430000,15.614079,115.9392,127.9527,10.361897,15.614079,strong effect
6,dubno_medium_4,math_start,6,1,589.778333,589.778333,0.000000,99.0422,99.0422,0.000000,0.000000,no noticeable effect
7,dubno_medium_5,math_start,6,1,386.136667,386.136667,0.000000,118.3218,118.3218,0.000000,0.000000,no noticeable effect
8,dubno_small,math_start,6,1,490.278333,490.278333,0.000000,114.6404,114.6404,0.000000,0.000000,no noticeable effect
9,dubno_small_1,math_start,6,1,310.613333,310.613333,0.000000,73.5916,73.5916,0.000000,0.000000,no noticeable effect



===== COMPACT SUMMARY: Dataset Ternopil-Dubno + simulation =====


,Dataset,Method,n_runs,n_unique_clusters_found,min_duration,max_duration,duration_range_pct_vs_min,min_distance,max_distance,distance_range_pct_vs_min,max_range_pct,seed_effect
0,dubno_gross_1,simulation_start,6,1,967.196667,967.196667,0.000000,143.1792,143.1792,0.000000,0.000000,no noticeable effect
1,dubno_gross_2,simulation_start,6,1,865.986667,865.986667,0.000000,141.8551,141.8551,0.000000,0.000000,no noticeable effect
2,dubno_gross_3,simulation_start,6,1,949.320000,949.320000,0.000000,140.5578,140.5578,0.000000,0.000000,no noticeable effect
3,dubno_medium_1,simulation_start,6,1,615.000000,615.000000,0.000000,113.7241,113.7241,0.000000,0.000000,no noticeable effect
4,dubno_medium_2,simulation_start,6,1,746.845000,746.845000,0.000000,124.8323,124.8323,0.000000,0.000000,no noticeable effect
5,dubno_medium_3,simulation_start,6,2,598.366667,751.398333,25.574898,115.9392,127.9527,10.361897,25.574898,strong effect
6,dubno_medium_4,simulation_start,6,1,615.000000,615.000000,0.000000,99.0422,99.0422,0.000000,0.000000,no noticeable effect
7,dubno_medium_5,simulation_start,6,1,403.161667,403.161667,0.000000,118.3218,118.3218,0.000000,0.000000,no noticeable effect
8,dubno_small,simulation_start,6,1,480.000000,480.000000,0.000000,116.7034,116.7034,0.000000,0.000000,no noticeable effect
9,dubno_small_1,simulation_start,6,1,371.000000,371.000000,0.000000,73.5916,73.5916,0.000000,0.000000,no noticeable effect


### The best result - K-medoids

In [219]:
fixed_seed = 42
best_iter_num = 10

In [ ]:
def run_clusters_kmedoids_best(cases, random_state, best_max_iter):
    results = []
    target_duration = 480

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"
        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=25,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        auto_result = run_kmedoids_clusterization(
            dataset=case,
            dist_km=dist_km,
            time_min=time_min,
            build_routes_fn=auto_alg.build_routes,
            random_state=random_state,
            max_iter=best_max_iter
        )

        sim_result = run_kmedoids_clusterization(
            dataset=case,
            dist_km=dist_km,
            time_min=time_min,
            build_routes_fn=sim_alg.build_routes,
            random_state=random_state,
            max_iter=best_max_iter,
            extra_kwargs={
                "start_from": "08:00",
                "start_to": "10:00",
                "step_min": 10,
                "target_duration": target_duration
            }
        )

        auto_summary = auto_result["summary"]
        sim_summary = sim_result["summary"]

        results.append({
            "Dataset": name,
            "Module": "KMedoids_Math_Start",
            "random_state": random_state,
            "max_iter": best_max_iter,
            "n_clusters": auto_result["n_clusters_final"],
            "n_routes": auto_summary["n_routes"],
            "total_distance_km": auto_summary["total_distance_km"],
            "total_duration_min": auto_summary["total_duration_min"],
            "total_served": auto_summary["total_served"],
            "silhouette_score": auto_result.get("silhouette_score", np.nan),
            "final_iterations": auto_result.get("num_iterations", np.nan),
        })

        results.append({
            "Dataset": name,
            "Module": "KMedoids_Simulate",
            "random_state": random_state,
            "max_iter": best_max_iter,
            "n_clusters": sim_result["n_clusters_final"],
            "n_routes": sim_summary["n_routes"],
            "total_distance_km": sim_summary["total_distance_km"],
            "total_duration_min": sim_summary["total_duration_min"],
            "total_served": sim_summary["total_served"],
            "silhouette_score": sim_result.get("silhouette_score", np.nan),
            "final_iterations": sim_result.get("num_iterations", np.nan),
        })

    return pd.DataFrame(results)

In [222]:
the_best_med_original = run_clusters_kmedoids_best(
    cases=cases,
    random_state=fixed_seed,
    best_max_iter=best_iter_num
)

In [223]:
the_best_med_simple = run_clusters_kmedoids_best(
    cases=cases_simple,
    random_state=fixed_seed,
    best_max_iter=best_iter_num
)

## GMM

In [25]:
from gmm_2 import find_optimal_n_clusters_gmm

In [38]:
def run_gmm_clusterization(
    dataset,
    dist_km,
    time_min,
    build_routes_fn,
    random_state=42,
    covariance_type="full",
    n_init=5,
    max_iter=200,
    extra_kwargs=None):
    '''
    GMM clusterization for one day for one algorithm
    '''
    if extra_kwargs is None:
        extra_kwargs = {}

    params = dataset["params"]

    build_routes_kwargs = {
        "max_crews": params["max_crews"],
        "max_workers": params["max_workers"],
        "workers_per_crew": params["workers_per_crew"],
        "max_route_duration_min": params["max_route_duration_min"],
        **extra_kwargs
    }

    result = find_optimal_n_clusters_gmm(
        day_stops=dataset["stops"],
        depo=dataset["depo"],
        dist_km=dist_km,
        time_min=time_min,
        max_crews=params["max_crews"],
        build_routes_gmm=build_routes_fn,
        build_routes_kwargs=build_routes_kwargs,
        random_state=random_state,
        covariance_type=covariance_type,
        n_init=n_init,
        max_iter=max_iter
    )

    labels = np.asarray(result["labels"])
    X = np.array([[s.lat, s.lon] for s in dataset["stops"]], dtype=float)

    n_points = len(labels)
    n_clusters_final = len(np.unique(labels)) if n_points > 0 else 0

    if n_points < 2 or n_clusters_final < 2 or n_clusters_final >= n_points:
        result["silhouette_score"] = np.nan
    else:
        result["silhouette_score"] = silhouette_score(X, labels)

    result["n_clusters_final"] = n_clusters_final
    result["n_ambiguous_points"] = len(result.get("ambiguous_info", {}))
    result["n_reassigned_points"] = sum(
        1 for x in result.get("reassignment_log", []) if x.get("moved", False)
    )

    result["bic"] = result.get("bic", np.nan)
    result["aic"] = result.get("aic", np.nan)
    result["log_likelihood"] = result.get("log_likelihood", np.nan)
    result["converged"] = result.get("converged", np.nan)
    result["n_iter_gmm"] = result.get("n_iter_gmm", np.nan)

    return result

In [39]:
def run_gmm_params_experiment(
    cases,
    build_routes_fn,
    method_name,
    covariance_types,
    random_state_values,
    max_iter_values,
    n_init_values,
    extra_kwargs=None):
    '''
    run GMM for different combinations of params
    '''
    if extra_kwargs is None:
        extra_kwargs = {}

    rows = []

    for case in cases:
        name = case["name"]
        current_cache = f"cache_routes/cache_{name}.json"

        coords = [(case["depo"].lat, case["depo"].lon)] + [
            (s.lat, s.lon) for s in case["stops"]
        ]

        dist_km, time_min = build_matrices(
            coords=coords,
            cache_path=current_cache,
            name=name,
            block_size=BLOCK_SIZE,
            osm_url=OSM_URL,
            osm_mode=OSM_MODE
        )

        for covariance_type in covariance_types:
            for random_state in random_state_values:
                for max_iter in max_iter_values:
                    for n_init in n_init_values:
                        try:
                            result = run_gmm_clusterization(
                                dataset=case,
                                dist_km=dist_km,
                                time_min=time_min,
                                build_routes_fn=build_routes_fn,
                                random_state=random_state,
                                covariance_type=covariance_type,
                                n_init=n_init,
                                max_iter=max_iter,
                                extra_kwargs=extra_kwargs
                            )

                            summary = result["summary"]

                            rows.append({
                                "Dataset": name,
                                "Method": method_name,
                                "covariance_type": covariance_type,
                                "random_state": random_state,
                                "max_iter": max_iter,
                                "n_init": n_init,
                                "status": "ok",

                                "n_clusters_final": result.get("n_clusters_final", np.nan),
                                "n_routes": summary.get("n_routes", np.nan),

                                "bic": result.get("bic", np.nan),
                                "aic": result.get("aic", np.nan),
                                "log_likelihood": result.get("log_likelihood", np.nan),
                                "converged": result.get("converged", np.nan),
                                "n_iter_gmm": result.get("n_iter_gmm", np.nan),

                                "total_duration_min": summary.get("total_duration_min", np.nan),
                                "total_distance_km": summary.get("total_distance_km", np.nan),
                                "total_served": summary.get("total_served", np.nan),

                                "silhouette_score": result.get("silhouette_score", np.nan),
                                "n_ambiguous_points": result.get("n_ambiguous_points", np.nan),
                                "n_reassigned_points": result.get("n_reassigned_points", np.nan),

                                "result_obj": result
                            })

                        except Exception as e:
                            rows.append({
                                "Dataset": name,
                                "Method": method_name,
                                "covariance_type": covariance_type,
                                "random_state": random_state,
                                "max_iter": max_iter,
                                "n_init": n_init,
                                "status": f"error: {e}",

                                "n_clusters_final": np.nan,
                                "n_routes": np.nan,

                                "bic": np.nan,
                                "aic": np.nan,
                                "log_likelihood": np.nan,
                                "converged": np.nan,
                                "n_iter_gmm": np.nan,

                                "total_duration_min": np.nan,
                                "total_distance_km": np.nan,
                                "total_served": np.nan,

                                "silhouette_score": np.nan,
                                "n_ambiguous_points": np.nan,
                                "n_reassigned_points": np.nan,

                                "result_obj": None
                            })

    return pd.DataFrame(rows)

In [ ]:
def summarize_gmm_experiment(df, group_cols=None):
    '''
    summary for experiments
    '''
    df_ok = df[df["status"] == "ok"].copy()

    if group_cols is None:
        group_cols = ["Dataset", "Method", "covariance_type", "max_iter", "n_init"]

    summary = (
        df_ok.groupby(group_cols)
        .agg(
            n_runs=("random_state", "count"),
            n_unique_seeds=("random_state", "nunique"),

            mean_duration=("total_duration_min", "mean"),
            std_duration=("total_duration_min", "std"),
            min_duration=("total_duration_min", "min"),
            max_duration=("total_duration_min", "max"),

            mean_distance=("total_distance_km", "mean"),
            std_distance=("total_distance_km", "std"),
            min_distance=("total_distance_km", "min"),
            max_distance=("total_distance_km", "max"),

            mean_silhouette=("silhouette_score", "mean"),
            std_silhouette=("silhouette_score", "std"),

            mean_bic=("bic", "mean"),
            min_bic=("bic", "min"),
            mean_aic=("aic", "mean"),

            mean_n_iter_gmm=("n_iter_gmm", "mean"),
            conv_rate=("converged", "mean"),

            mean_ambiguous=("n_ambiguous_points", "mean"),
            mean_reassigned=("n_reassigned_points", "mean"),
        )
        .reset_index()
    )

    summary["duration_range_abs"] = summary["max_duration"] - summary["min_duration"]
    summary["distance_range_abs"] = summary["max_distance"] - summary["min_distance"]

    summary["duration_cv_pct"] = np.where(
        summary["mean_duration"] > 0,
        summary["std_duration"] / summary["mean_duration"] * 100,
        np.nan
    )

    summary["distance_cv_pct"] = np.where(
        summary["mean_distance"] > 0,
        summary["std_distance"] / summary["mean_distance"] * 100,
        np.nan
    )

    return summary.sort_values(group_cols)

In [45]:
def select_best_gmm_run(group):
    '''
    select best for one case
    '''
    group = group[group["status"] == "ok"].copy()

    if group.empty:
        return None

    group["silhouette_for_sort"] = group["silhouette_score"].fillna(-999999)

    group = group.sort_values(
        by=[
            "total_duration_min",
            "total_distance_km",
            "bic",
            "aic",
            "silhouette_for_sort"
        ],
        ascending=[True, True, True, True, False]
    )

    return group.iloc[0]

In [42]:
def find_best_gmm_params_for_cases(
    cases,
    build_routes_fn,
    method_name,
    covariance_types,
    random_state_values,
    max_iter_values,
    n_init_values,
    extra_kwargs=None):
    '''
    Run for whole dataset
    '''
    all_runs_df = run_gmm_params_experiment(
        cases=cases,
        build_routes_fn=build_routes_fn,
        method_name=method_name,
        covariance_types=covariance_types,
        random_state_values=random_state_values,
        max_iter_values=max_iter_values,
        n_init_values=n_init_values,
        extra_kwargs=extra_kwargs
    )

    best_rows = []

    for (dataset, method), group in all_runs_df.groupby(["Dataset", "Method"]):
        best_row = select_best_gmm_run(group)
        if best_row is not None:
            best_rows.append(best_row)

    best_runs_df = pd.DataFrame(best_rows).reset_index(drop=True)

    return all_runs_df, best_runs_df

In [104]:
covariance_types = ["full", "tied", "diag"]
seed_values = [5, 20, 42, 100]
max_iter_values=[100, 300, 500]
n_init_values=[3, 5, 10]

In [51]:
gmm_all_auto, gmm_best_auto = find_best_gmm_params_for_cases(
    cases=cases,
    build_routes_fn=auto_alg.build_routes,
    method_name="GMM_Math_Start",
    covariance_types=covariance_types,
    random_state_values=seed_values,
    max_iter_values=max_iter_values,
    n_init_values=n_init_values
)

In [55]:
gmm_all_sim, gmm_best_sim = find_best_gmm_params_for_cases(
    cases=cases,
    build_routes_fn=sim_alg.build_routes,
    method_name="GMM_Simulate",
    covariance_types=covariance_types,
    random_state_values=seed_values,
    max_iter_values=max_iter_values,
    n_init_values=n_init_values,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [56]:
gmm_all_auto_simple, gmm_best_auto_simple = find_best_gmm_params_for_cases(
    cases=cases_simple,
    build_routes_fn=auto_alg.build_routes,
    method_name="GMM_Math_Start_simple",
    covariance_types=covariance_types,
    random_state_values=seed_values,
    max_iter_values=max_iter_values,
    n_init_values=n_init_values
)

In [57]:
gmm_all_sim_simple, gmm_best_sim_simple = find_best_gmm_params_for_cases(
    cases=cases_simple,
    build_routes_fn=sim_alg.build_routes,
    method_name="GMM_Simulate",
    covariance_types=covariance_types,
    random_state_values=seed_values,
    max_iter_values=max_iter_values,
    n_init_values=n_init_values,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [273]:
def build_labels_from_clusters(clusters, n_points):
    labels = np.full(n_points, -1, dtype=int)

    for cluster_id, point_indices in clusters.items():
        for point_idx in point_indices:
            labels[point_idx] = int(cluster_id)

    if np.any(labels == -1):
        raise ValueError("Some points were not assigned to any cluster")

    return labels


def recalc_gmm_metrics_from_result_obj(df, cases_dict):
    df = df.copy()

    new_n_clusters = []
    new_silhouette = []
    new_n_ambiguous = []
    new_n_reassigned = []

    for _, row in df.iterrows():
        result = row.get("result_obj", None)
        dataset_name = row["Dataset"]

        if result is None:
            new_n_clusters.append(np.nan)
            new_silhouette.append(np.nan)
            new_n_ambiguous.append(np.nan)
            new_n_reassigned.append(np.nan)
            continue

        case = cases_dict[dataset_name]
        n_points = len(case["stops"])

        clusters_final = result.get("clusters_final", None)
        if clusters_final is None:
            new_n_clusters.append(np.nan)
            new_silhouette.append(np.nan)
            new_n_ambiguous.append(np.nan)
            new_n_reassigned.append(np.nan)
            continue

        labels_final = build_labels_from_clusters(clusters_final, n_points)
        result["labels_final"] = labels_final

        n_clusters_final = len(np.unique(labels_final)) if n_points > 0 else 0

        X = np.array([[s.lat, s.lon] for s in case["stops"]], dtype=float)

        if n_points < 2 or n_clusters_final < 2 or n_clusters_final >= n_points:
            silhouette_val = np.nan
        else:
            silhouette_val = silhouette_score(X, labels_final)

        result["n_clusters_final"] = n_clusters_final
        result["silhouette_score"] = silhouette_val
        result["n_ambiguous_points"] = len(result.get("ambiguous_info", {}))
        result["n_reassigned_points"] = sum(
            1 for x in result.get("reassignment_log", []) if x.get("moved", False)
        )

        new_n_clusters.append(n_clusters_final)
        new_silhouette.append(silhouette_val)
        new_n_ambiguous.append(result["n_ambiguous_points"])
        new_n_reassigned.append(result["n_reassigned_points"])

    df["n_clusters_final"] = new_n_clusters
    df["silhouette_score"] = new_silhouette
    df["n_ambiguous_points"] = new_n_ambiguous
    df["n_reassigned_points"] = new_n_reassigned

    return df

In [274]:
cases_dict = {case["name"]: case for case in cases}
cases_simple_dict = {case["name"]: case for case in cases_simple}

In [275]:
gmm_all_auto = recalc_gmm_metrics_from_result_obj(gmm_all_auto, cases_dict)
gmm_all_sim = recalc_gmm_metrics_from_result_obj(gmm_all_sim, cases_dict)

gmm_all_auto_simple = recalc_gmm_metrics_from_result_obj(gmm_all_auto_simple, cases_simple_dict)
gmm_all_sim_simple = recalc_gmm_metrics_from_result_obj(gmm_all_sim_simple, cases_simple_dict)

### Interpretation of the GMM results 

In [276]:
all_runs_df = pd.concat([gmm_all_auto, gmm_all_sim, gmm_all_auto_simple, gmm_all_sim_simple], ignore_index=True)

In [277]:
all_runs_clean = all_runs_df[
    all_runs_df["random_state"].isin(seed_values)
].copy()

In [278]:
all_runs_clean

,Dataset,Method,covariance_type,random_state,max_iter,n_init,status,n_clusters_final,n_routes,bic,...,log_likelihood,converged,n_iter_gmm,total_duration_min,total_distance_km,total_served,silhouette_score,n_ambiguous_points,n_reassigned_points,result_obj
0,Kyiv_small,GMM_Math_Start,full,5,100,3,ok,13,18,-1393.630995,...,2.291272,True,29,11632.308333,2541.0598,405,0.436065,12,8,"{'model': GaussianMixture(n_components=13, n_i..."
1,Kyiv_small,GMM_Math_Start,full,5,100,5,ok,13,18,-1393.630995,...,2.291272,True,29,11632.308333,2541.0598,405,0.436065,12,8,"{'model': GaussianMixture(n_components=13, n_i..."
2,Kyiv_small,GMM_Math_Start,full,5,100,10,ok,13,18,-1393.630995,...,2.291272,True,29,11632.308333,2541.0598,405,0.436065,12,8,"{'model': GaussianMixture(n_components=13, n_i..."
3,Kyiv_small,GMM_Math_Start,full,5,300,3,ok,13,18,-1393.630995,...,2.291272,True,29,11632.308333,2541.0598,405,0.436065,12,8,"{'model': GaussianMixture(max_iter=300, n_comp..."
4,Kyiv_small,GMM_Math_Start,full,5,300,5,ok,13,18,-1393.630995,...,2.291272,True,29,11632.308333,2541.0598,405,0.436065,12,8,"{'model': GaussianMixture(max_iter=300, n_comp..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11875,dubno_gross_3,GMM_Simulate,diag,100,300,5,ok,1,2,-505.242062,...,4.495550,True,2,949.320000,140.5578,58,NaN,0,0,{'model': GaussianMixture(covariance_type='dia...
11876,dubno_gross_3,GMM_Simulate,diag,100,300,10,ok,1,2,-505.242062,...,4.495550,True,2,949.320000,140.5578,58,NaN,0,0,{'model': GaussianMixture(covariance_type='dia...
11877,dubno_gross_3,GMM_Simulate,diag,100,500,3,ok,1,2,-505.242062,...,4.495550,True,2,949.320000,140.5578,58,NaN,0,0,{'model': GaussianMixture(covariance_type='dia...
11878,dubno_gross_3,GMM_Simulate,diag,100,500,5,ok,1,2,-505.242062,...,4.495550,True,2,949.320000,140.5578,58,NaN,0,0,{'model': GaussianMixture(covariance_type='dia...


In [279]:
all_runs_clean["Method"] = all_runs_clean["Method"].replace({
    "GMM_Math_Start_simple": "GMM_Math_Start"
})

In [280]:
per_dataset_summary = (
    all_runs_clean
    .groupby(
        ["Dataset", "Method", "covariance_type", "max_iter", "n_init"],
        as_index=False
    )
    .agg(
        mean_bic=("bic", "mean"),
        std_bic=("bic", "std"),
        mean_log_likelihood=("log_likelihood", "mean"),
        mean_silhouette_score=("silhouette_score", "mean"),
        n_successful_runs=("random_state", "count"),
        conv_rate=("converged", "mean"),
        mean_n_iter_gmm=("n_iter_gmm", "mean"),
        max_n_iter_gmm=("n_iter_gmm", "max"),
    )
)

In [281]:
per_dataset_summary

,Dataset,Method,covariance_type,max_iter,n_init,mean_bic,std_bic,mean_log_likelihood,mean_silhouette_score,n_successful_runs,conv_rate,mean_n_iter_gmm,max_n_iter_gmm
0,Kyiv_gross_1,GMM_Math_Start,diag,100,3,-2187.774769,167.726472,2.044796,0.448531,4,1.0,13.25,23
1,Kyiv_gross_1,GMM_Math_Start,diag,100,5,-2144.518435,217.954847,1.973901,0.450387,4,1.0,12.75,23
2,Kyiv_gross_1,GMM_Math_Start,diag,100,10,-2288.619975,84.153351,2.158031,0.438858,4,1.0,17.50,22
3,Kyiv_gross_1,GMM_Math_Start,diag,300,3,-2187.774769,167.726472,2.044796,0.448531,4,1.0,13.25,23
4,Kyiv_gross_1,GMM_Math_Start,diag,300,5,-2144.518435,217.954847,1.973901,0.450387,4,1.0,12.75,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2155,ternopil_small_1,GMM_Simulate,tied,300,5,-1026.453492,0.000000,3.505022,NaN,6,1.0,2.00,2
2156,ternopil_small_1,GMM_Simulate,tied,300,10,-1026.453492,0.000000,3.505022,NaN,6,1.0,2.00,2
2157,ternopil_small_1,GMM_Simulate,tied,500,3,-1026.453492,0.000000,3.505022,NaN,6,1.0,2.00,2
2158,ternopil_small_1,GMM_Simulate,tied,500,5,-1026.453492,0.000000,3.505022,NaN,6,1.0,2.00,2


In [282]:
global_summary = (
    per_dataset_summary
    .groupby(
        ["Method", "covariance_type", "max_iter", "n_init"],
        as_index=False
    )
    .agg(
        mean_bic=("mean_bic", "mean"),
        std_bic_across_datasets=("mean_bic", "std"),
        mean_log_likelihood=("mean_log_likelihood", "mean"),
        mean_silhouette_score=("mean_silhouette_score", "mean"),
        datasets_covered=("Dataset", "count"),
        n_successful_runs=("n_successful_runs", "sum"),
        conv_rate=("conv_rate", "mean"),
        mean_n_iter_gmm=("mean_n_iter_gmm", "mean"),
        max_n_iter_gmm=("max_n_iter_gmm", "max")
    )
)

In [283]:
global_summary

,Method,covariance_type,max_iter,n_init,mean_bic,std_bic_across_datasets,mean_log_likelihood,mean_silhouette_score,datasets_covered,n_successful_runs,conv_rate,mean_n_iter_gmm,max_n_iter_gmm
0,GMM_Math_Start,diag,100,3,-985.100678,742.227386,3.102338,0.380575,40,200,1.0,10.060417,37
1,GMM_Math_Start,diag,100,5,-986.573595,740.580144,3.123197,0.380496,40,200,1.0,10.787500,50
2,GMM_Math_Start,diag,100,10,-990.175554,743.171641,3.147415,0.366266,40,200,1.0,11.793750,50
3,GMM_Math_Start,diag,300,3,-985.100678,742.227386,3.102338,0.380575,40,200,1.0,10.060417,37
4,GMM_Math_Start,diag,300,5,-986.573595,740.580144,3.123197,0.380496,40,200,1.0,10.787500,50
5,GMM_Math_Start,diag,300,10,-990.175554,743.171641,3.147415,0.366266,40,200,1.0,11.793750,50
6,GMM_Math_Start,diag,500,3,-985.100678,742.227386,3.102338,0.380575,40,200,1.0,10.060417,37
7,GMM_Math_Start,diag,500,5,-986.573595,740.580144,3.123197,0.380496,40,200,1.0,10.787500,50
8,GMM_Math_Start,diag,500,10,-990.175554,743.171641,3.147415,0.366266,40,200,1.0,11.793750,50
9,GMM_Math_Start,full,100,3,-983.717886,751.908070,3.161115,0.394675,40,200,1.0,11.593750,50


In [284]:
top_math = (
    global_summary[
        (global_summary["Method"] == "GMM_Math_Start") &
        (global_summary["max_iter"] == 100)
    ]
    .sort_values(by=["mean_bic", "std_bic_across_datasets"], ascending=[True, True])
)

In [285]:
top_math

,Method,covariance_type,max_iter,n_init,mean_bic,std_bic_across_datasets,mean_log_likelihood,mean_silhouette_score,datasets_covered,n_successful_runs,conv_rate,mean_n_iter_gmm,max_n_iter_gmm
2,GMM_Math_Start,diag,100,10,-990.175554,743.171641,3.147415,0.366266,40,200,1.0,11.793750,50
1,GMM_Math_Start,diag,100,5,-986.573595,740.580144,3.123197,0.380496,40,200,1.0,10.787500,50
11,GMM_Math_Start,full,100,10,-985.836887,751.845680,3.178686,0.387640,40,200,1.0,12.222917,50
0,GMM_Math_Start,diag,100,3,-985.100678,742.227386,3.102338,0.380575,40,200,1.0,10.060417,37
9,GMM_Math_Start,full,100,3,-983.717886,751.908070,3.161115,0.394675,40,200,1.0,11.593750,50
10,GMM_Math_Start,full,100,5,-980.624637,745.820020,3.169173,0.394514,40,200,1.0,11.695833,50
20,GMM_Math_Start,tied,100,10,-963.377540,726.288384,3.035199,0.458204,40,200,1.0,6.479167,32
19,GMM_Math_Start,tied,100,5,-961.776782,724.651097,3.033956,0.465873,40,200,1.0,6.054167,33
18,GMM_Math_Start,tied,100,3,-960.055253,723.892296,3.029141,0.465112,40,200,1.0,6.035417,32


In [286]:
top_sim = (
    global_summary[
        (global_summary["Method"] == "GMM_Simulate") &
        (global_summary["max_iter"] == 100)
    ]
    .sort_values(by=["mean_bic", "std_bic_across_datasets"], ascending=[True, True])
)

In [287]:
top_sim

,Method,covariance_type,max_iter,n_init,mean_bic,std_bic_across_datasets,mean_log_likelihood,mean_silhouette_score,datasets_covered,n_successful_runs,conv_rate,mean_n_iter_gmm,max_n_iter_gmm
29,GMM_Simulate,diag,100,10,-1001.041362,765.541591,3.132557,0.372055,40,240,1.0,11.633333,50
28,GMM_Simulate,diag,100,5,-998.756530,764.477731,3.116128,0.385309,40,240,1.0,11.025000,50
27,GMM_Simulate,diag,100,3,-997.749989,764.829946,3.102728,0.386282,40,240,1.0,10.283333,39
38,GMM_Simulate,full,100,10,-990.194491,760.784605,3.166448,0.372124,40,240,1.0,12.166667,50
37,GMM_Simulate,full,100,5,-987.657205,758.435789,3.158644,0.378908,40,240,1.0,11.625000,50
36,GMM_Simulate,full,100,3,-986.090358,757.107894,3.147461,0.380062,40,240,1.0,11.366667,50
47,GMM_Simulate,tied,100,10,-977.541151,746.029074,3.054539,0.461053,40,240,1.0,6.608333,32
46,GMM_Simulate,tied,100,5,-976.588033,745.422243,3.051664,0.470386,40,240,1.0,6.195833,33
45,GMM_Simulate,tied,100,3,-975.795957,744.694107,3.049832,0.469995,40,240,1.0,6.283333,32


### The best result - GMM

In [291]:
the_best_covariance = ['diag']
the_best_max_iter = [100]
the_best_n_init = [10]
fixed_seed = [42]

In [293]:
gmm_math_final_original = run_gmm_params_experiment(
    cases=cases,
    build_routes_fn=auto_alg.build_routes,
    method_name="GMM_Math_Start",
    covariance_types=the_best_covariance,
    max_iter_values=the_best_max_iter,
    n_init_values=the_best_n_init,
    random_state_values=fixed_seed
)

In [294]:
gmm_sim_final_original = run_gmm_params_experiment(
    cases=cases,
    build_routes_fn=sim_alg.build_routes,
    method_name="GMM_Simulate",
    covariance_types=the_best_covariance,
    max_iter_values=the_best_max_iter,
    n_init_values=the_best_n_init,
    random_state_values=fixed_seed,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [295]:
gmm_math_final = run_gmm_params_experiment(
    cases=cases_simple,
    build_routes_fn=auto_alg.build_routes,
    method_name="GMM_Math_Start",
    covariance_types=the_best_covariance,
    max_iter_values=the_best_max_iter,
    n_init_values=the_best_n_init,
    random_state_values=fixed_seed
)

In [296]:
gmm_sim_final = run_gmm_params_experiment(
    cases=cases_simple,
    build_routes_fn=sim_alg.build_routes,
    method_name="GMM_Simulate",
    covariance_types=the_best_covariance,
    max_iter_values=the_best_max_iter,
    n_init_values=the_best_n_init,
    random_state_values=fixed_seed,
    extra_kwargs={
        "start_from": "08:00",
        "start_to": "10:00",
        "step_min": 10,
        "target_duration": 480
    }
)

In [297]:
gmm_math_final_original = recalc_gmm_metrics_from_result_obj(gmm_math_final_original, cases_dict)
gmm_sim_final_original = recalc_gmm_metrics_from_result_obj(gmm_sim_final_original, cases_dict)

gmm_math_final = recalc_gmm_metrics_from_result_obj(gmm_math_final, cases_simple_dict)
gmm_sim_final = recalc_gmm_metrics_from_result_obj(gmm_sim_final, cases_simple_dict)

In [298]:
gmm_original = pd.concat(
    [gmm_math_final_original, gmm_sim_final_original],
    ignore_index=True
)

gmm_simple = pd.concat(
    [gmm_math_final, gmm_sim_final],
    ignore_index=True
)

### The best clasterization

In [299]:
def split_module_name(name):
    mapping = {
        "Agglo_Math_Start": ("Agglomerative", "Math_Start"),
        "Agglo_Simulate": ("Agglomerative", "Simulate"),
        "KMedoids_Math_Start": ("KMedoids", "Math_Start"),
        "KMedoids_Simulate": ("KMedoids", "Simulate"),
        "GMM_Math_Start": ("GMM", "Math_Start"),
        "GMM_Simulate": ("GMM", "Simulate"),
    }
    return mapping.get(name, (None, None))

In [300]:
def prepare_agglo(df, dataset_type):
    temp = df.copy()

    temp[["Clustering_Method", "Routing_Method"]] = (
        temp["Module"].apply(split_module_name).apply(pd.Series)
    )

    temp = temp.rename(columns={
        "n_clusters": "n_clusters"
    })

    temp["Dataset_Type"] = dataset_type

    temp = temp[
        [
            "Dataset",
            "Dataset_Type",
            "Clustering_Method",
            "Routing_Method",
            "n_clusters",
            "n_routes",
            "total_distance_km",
            "total_duration_min",
            "total_served",
            "silhouette_score"
        ]
    ].copy()

    return temp

In [302]:
def prepare_kmedoids(df, dataset_type):
    temp = df.copy()

    temp[["Clustering_Method", "Routing_Method"]] = (
        temp["Module"].apply(split_module_name).apply(pd.Series)
    )

    temp["Dataset_Type"] = dataset_type

    temp = temp[
        [
            "Dataset",
            "Dataset_Type",
            "Clustering_Method",
            "Routing_Method",
            "n_clusters",
            "n_routes",
            "total_distance_km",
            "total_duration_min",
            "total_served",
            "silhouette_score"
        ]
    ].copy()

    return temp

In [303]:
def prepare_gmm(df, dataset_type):
    temp = df.copy()

    temp = temp.rename(columns={
        "Method": "Module",
        "n_clusters_final": "n_clusters"
    })

    temp[["Clustering_Method", "Routing_Method"]] = (
        temp["Module"].apply(split_module_name).apply(pd.Series)
    )

    temp["Dataset_Type"] = dataset_type

    temp = temp[
        [
            "Dataset",
            "Dataset_Type",
            "Clustering_Method",
            "Routing_Method",
            "n_clusters",
            "n_routes",
            "total_distance_km",
            "total_duration_min",
            "total_served",
            "silhouette_score"
        ]
    ].copy()

    return temp

In [304]:
agglo_original_prepared = prepare_agglo(the_best_ag_res_original, "original")
agglo_simple_prepared = prepare_agglo(the_best_ag_res_simple, "simple")

In [305]:
kmed_original_prepared = prepare_kmedoids(the_best_med_original, "original")
kmed_simple_prepared = prepare_kmedoids(the_best_med_simple, "simple")

In [306]:
gmm_original_prepared = prepare_gmm(gmm_original, "original")
gmm_simple_prepared = prepare_gmm(gmm_simple, "simple")

In [307]:
all_clustering_methods = all_results = pd.concat([agglo_original_prepared, agglo_simple_prepared, kmed_original_prepared, kmed_simple_prepared, gmm_original_prepared, gmm_simple_prepared], ignore_index=True)

In [316]:
summary_compare = (
    all_results
    .groupby(["Dataset_Type", "Routing_Method", "Clustering_Method"], as_index=False)
    .agg(
        mean_duration=("total_duration_min", "mean"),
        std_duration=("total_duration_min", "std"),
        med_duration=("total_duration_min", "median"),
        mean_distance=("total_distance_km", "mean"),
        std_distance=("total_distance_km", "std"),
        med_distance=("total_distance_km", "median"),
        mean_routes=("n_routes", "mean"),
        mean_clusters=("n_clusters", "mean"),
        mean_silhouette_score=("silhouette_score", "mean")
    )
    .sort_values(
        by=["Dataset_Type", "Routing_Method", "mean_duration", "mean_distance"],
        ascending=[True, True, True, True]
    )
)

In [317]:
summary_compare

,Dataset_Type,Routing_Method,Clustering_Method,mean_duration,std_duration,med_duration,mean_distance,std_distance,med_distance,mean_routes,mean_clusters,mean_silhouette_score
2,original,Math_Start,KMedoids,6504.993167,5812.432188,5494.331667,1585.280825,1223.294565,1453.45705,9.80,3.40,0.400423
0,original,Math_Start,Agglomerative,6663.386833,5913.333653,5846.536667,1602.805450,1225.829399,1405.07555,10.05,3.65,0.392876
1,original,Math_Start,GMM,6876.871250,6101.913994,5560.317500,1642.419450,1249.958507,1449.98940,10.60,5.20,0.408128
5,original,Simulate,KMedoids,5289.987250,4548.514561,4274.530000,1497.180050,1127.965245,1358.71735,8.95,4.80,0.406321
3,original,Simulate,Agglomerative,5654.343750,4873.477756,4848.945833,1522.877515,1149.136835,1436.92820,9.40,4.05,0.404538
4,original,Simulate,GMM,5875.262833,5032.215518,4824.526667,1598.878165,1206.513579,1416.00170,10.25,5.85,0.400859
6,simple,Math_Start,Agglomerative,2303.372417,1826.075215,1834.150833,327.686650,232.863757,264.58920,3.80,1.85,0.328375
8,simple,Math_Start,KMedoids,2356.827500,1852.369046,1854.190000,335.633195,238.880903,266.23300,3.90,1.70,0.302672
7,simple,Math_Start,GMM,2618.892000,2142.679708,1854.190000,328.572490,226.275290,266.23300,4.55,2.90,0.381686
9,simple,Simulate,Agglomerative,2103.653750,1619.201650,1476.240833,328.348505,229.137780,262.67775,3.75,2.00,0.336951


In [319]:
summary_compare["cv_duration"] = summary_compare["std_duration"] / summary_compare["mean_duration"]

In [320]:
summary_compare

,Dataset_Type,Routing_Method,Clustering_Method,mean_duration,std_duration,med_duration,mean_distance,std_distance,med_distance,mean_routes,mean_clusters,mean_silhouette_score,cv_duration
2,original,Math_Start,KMedoids,6504.993167,5812.432188,5494.331667,1585.280825,1223.294565,1453.45705,9.80,3.40,0.400423,0.893534
0,original,Math_Start,Agglomerative,6663.386833,5913.333653,5846.536667,1602.805450,1225.829399,1405.07555,10.05,3.65,0.392876,0.887437
1,original,Math_Start,GMM,6876.871250,6101.913994,5560.317500,1642.419450,1249.958507,1449.98940,10.60,5.20,0.408128,0.887310
5,original,Simulate,KMedoids,5289.987250,4548.514561,4274.530000,1497.180050,1127.965245,1358.71735,8.95,4.80,0.406321,0.859835
3,original,Simulate,Agglomerative,5654.343750,4873.477756,4848.945833,1522.877515,1149.136835,1436.92820,9.40,4.05,0.404538,0.861900
4,original,Simulate,GMM,5875.262833,5032.215518,4824.526667,1598.878165,1206.513579,1416.00170,10.25,5.85,0.400859,0.856509
6,simple,Math_Start,Agglomerative,2303.372417,1826.075215,1834.150833,327.686650,232.863757,264.58920,3.80,1.85,0.328375,0.792783
8,simple,Math_Start,KMedoids,2356.827500,1852.369046,1854.190000,335.633195,238.880903,266.23300,3.90,1.70,0.302672,0.785959
7,simple,Math_Start,GMM,2618.892000,2142.679708,1854.190000,328.572490,226.275290,266.23300,4.55,2.90,0.381686,0.818163
9,simple,Simulate,Agglomerative,2103.653750,1619.201650,1476.240833,328.348505,229.137780,262.67775,3.75,2.00,0.336951,0.769709


In [321]:
best_per_group = (
    summary_compare
    .groupby(["Dataset_Type", "Routing_Method"], as_index=False)
    .first()
)

best_per_group

,Dataset_Type,Routing_Method,Clustering_Method,mean_duration,std_duration,med_duration,mean_distance,std_distance,med_distance,mean_routes,mean_clusters,mean_silhouette_score,cv_duration
0,original,Math_Start,KMedoids,6504.993167,5812.432188,5494.331667,1585.280825,1223.294565,1453.45705,9.80,3.40,0.400423,0.893534
1,original,Simulate,KMedoids,5289.987250,4548.514561,4274.530000,1497.180050,1127.965245,1358.71735,8.95,4.80,0.406321,0.859835
2,simple,Math_Start,Agglomerative,2303.372417,1826.075215,1834.150833,327.686650,232.863757,264.58920,3.80,1.85,0.328375,0.792783
3,simple,Simulate,Agglomerative,2103.653750,1619.201650,1476.240833,328.348505,229.137780,262.67775,3.75,2.00,0.336951,0.769709


In [328]:
ranked_summary = (
    summary_compare
    .sort_values(
        by=["Dataset_Type", "Routing_Method", "mean_duration", "mean_distance"],
        ascending=[True, True, True, True]
    )
    .copy()
)

ranked_summary["rank"] = (
    ranked_summary
    .groupby(["Dataset_Type", "Routing_Method"])
    .cumcount() + 1
)

In [327]:
ranked_summary

,Dataset_Type,Routing_Method,Clustering_Method,mean_duration,std_duration,med_duration,mean_distance,std_distance,med_distance,mean_routes,mean_clusters,mean_silhouette_score,cv_duration,rank
2,original,Math_Start,KMedoids,6504.993167,5812.432188,5494.331667,1585.280825,1223.294565,1453.45705,9.80,3.40,0.400423,0.893534,1
0,original,Math_Start,Agglomerative,6663.386833,5913.333653,5846.536667,1602.805450,1225.829399,1405.07555,10.05,3.65,0.392876,0.887437,2
1,original,Math_Start,GMM,6876.871250,6101.913994,5560.317500,1642.419450,1249.958507,1449.98940,10.60,5.20,0.408128,0.887310,3
5,original,Simulate,KMedoids,5289.987250,4548.514561,4274.530000,1497.180050,1127.965245,1358.71735,8.95,4.80,0.406321,0.859835,1
3,original,Simulate,Agglomerative,5654.343750,4873.477756,4848.945833,1522.877515,1149.136835,1436.92820,9.40,4.05,0.404538,0.861900,2
4,original,Simulate,GMM,5875.262833,5032.215518,4824.526667,1598.878165,1206.513579,1416.00170,10.25,5.85,0.400859,0.856509,3
6,simple,Math_Start,Agglomerative,2303.372417,1826.075215,1834.150833,327.686650,232.863757,264.58920,3.80,1.85,0.328375,0.792783,1
8,simple,Math_Start,KMedoids,2356.827500,1852.369046,1854.190000,335.633195,238.880903,266.23300,3.90,1.70,0.302672,0.785959,2
7,simple,Math_Start,GMM,2618.892000,2142.679708,1854.190000,328.572490,226.275290,266.23300,4.55,2.90,0.381686,0.818163,3
9,simple,Simulate,Agglomerative,2103.653750,1619.201650,1476.240833,328.348505,229.137780,262.67775,3.75,2.00,0.336951,0.769709,1


In [325]:
average_rank = (
    ranked_summary
    .groupby("Clustering_Method", as_index=False)
    .agg(
        average_rank=("rank", "mean"),
        first_places=("rank", lambda x: (x == 1).sum()),
        mean_of_mean_duration=("mean_duration", "mean"),
        mean_of_mean_distance=("mean_distance", "mean"),
        mean_cv_duration=("cv_duration", "mean")
    )
    .sort_values(
        by=["average_rank", "first_places", "mean_of_mean_duration"],
        ascending=[True, False, True]
    )
)

In [326]:
average_rank

,Clustering_Method,average_rank,first_places,mean_of_mean_duration,mean_of_mean_distance,mean_cv_duration
2,KMedoids,1.5,2,4070.434083,937.011064,0.823821
0,Agglomerative,1.5,2,4181.189188,945.429530,0.827957
1,GMM,3.0,0,4443.986417,973.924734,0.836116


### Friedman test

In [336]:
from scipy.stats import friedmanchisquare
from scipy.stats import wilcoxon
from itertools import combinations

In [337]:
duration_pivot = all_clustering_methods.pivot_table(
    index=["Dataset", "Dataset_Type", "Routing_Method"],
    columns="Clustering_Method",
    values="total_duration_min"
)

duration_pivot.head()

Clustering_Method                         Agglomerative           GMM  \
Dataset      Dataset_Type Routing_Method                                
Kyiv_gross_1 original     Math_Start       14184.416667  13631.273333   
                          Simulate         11400.778333  11457.753333   
Kyiv_gross_2 original     Math_Start       14032.835000  14123.510000   
                          Simulate         11953.331667  11946.373333   
Kyiv_gross_3 original     Math_Start       13292.070000  14004.356667   

Clustering_Method                             KMedoids  
Dataset      Dataset_Type Routing_Method                
Kyiv_gross_1 original     Math_Start      14152.805000  
                          Simulate        11586.650000  
Kyiv_gross_2 original     Math_Start      14291.818333  
                          Simulate        11101.686667  
Kyiv_gross_3 original     Math_Start      12926.500000

In [338]:
stat, p_value = friedmanchisquare(
    duration_pivot["Agglomerative"],
    duration_pivot["KMedoids"],
    duration_pivot["GMM"]
)

print(f"Friedman statistic = {stat:.4f}")
print(f"p-value = {p_value}")

Friedman statistic = 37.2919
p-value = 7.983148021588685e-09


In [339]:
methods = ["Agglomerative", "KMedoids", "GMM"]

pairwise_results = []

for m1, m2 in combinations(methods, 2):
    stat, p = wilcoxon(duration_pivot[m1], duration_pivot[m2])
    pairwise_results.append((m1, m2, stat, p))

pairwise_results

[('Agglomerative', 'KMedoids', 419.0, 0.054254195013277606),
 ('Agglomerative', 'GMM', 187.0, 1.7271867114709443e-06),
 ('KMedoids', 'GMM', 123.0, 6.813275396349823e-07)]

In [341]:
rank_df = duration_pivot.rank(axis=1, method="average", ascending=True)
average_ranks = rank_df.mean().sort_values()
print(average_ranks)

Clustering_Method
KMedoids         1.75625
Agglomerative    1.79375
GMM              2.45000
dtype: float64
